# 06 — Embedded storage: the default IPFS substitute

The gen2 replacement for the external `ipfrs-core` dependency: `hllset-cid`
(embedded SHA-1 content-addressed identifiers) plus `hllset-storage` with
`SledStorage` as the embedded default backend. No external daemon, no
network, no path outside the collection — and a full ipfrs/IPFS CID backend
can be plugged in later behind the same `Cid` type.


In [2]:
:dep hllset-contracts = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-contracts" }
:dep hllset-cid = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-cid" }
:dep hllset-storage = { path = "/home/alexmy/SGS/SGS_lib/fractal_manifold_gen2/hllset-next-v2/crates/hllset-storage" }


In [3]:
use hllset_cid::Cid;
use hllset_contracts::sha1_hex;
use hllset_storage::{MemoryStorage, SledStorage, Storage};
println!("embedded storage + CID loaded");


embedded storage + CID loaded


---
## The CID: content is identity

A `Cid` is the SHA-1 of the bytes, displayed as `h:<sha1>` — the same
identity law as the rest of the algebra.


In [4]:
let data = b"hello world";
let cid: Cid = Cid::compute(data);
println!("data:     {:?}", std::str::from_utf8(data).unwrap());
println!("cid:      {} ({} chars)", cid, cid.to_string().len());
println!("prefixed: {}", cid.prefixed());
println!("matches contracts sha1:      {}", cid.as_str() == sha1_hex(data));
println!("same data -> same CID:       {}", Cid::compute(data) == cid);
println!("different data -> different:  {}", Cid::compute(b"hello World") != cid);


data:     "hello world"
cid:      h:2aae6c35c94fcfb415dbe95f408b9ce91ee846ed (42 chars)
prefixed: h:2aae6c35c94fcfb415dbe95f408b9ce91ee846ed
matches contracts sha1:      true
same data -> same CID:       true
different data -> different:  true


---
## MemoryStorage: the dev backend

In-memory, for development and testing.


In [5]:
let mem: MemoryStorage = MemoryStorage::new();
mem.store("h:alpha", b"a").unwrap();
mem.store("h:beta", b"b").unwrap();
println!("load h:alpha: {:?}", mem.load("h:alpha").unwrap());
println!("exists h:beta: {}", mem.exists("h:beta").unwrap());
println!("list h:*: {:?}", mem.list("h:").unwrap());


load h:alpha: Some([97])
exists h:beta: true
list h:*: ["h:alpha", "h:beta"]


---
## SledStorage: the embedded default

sled on disk (temp DB here), content keys, no external service.


In [6]:
let sled: SledStorage = SledStorage::open_temp().unwrap();
sled.store("h:alpha", b"a").unwrap();
sled.store("h:beta", b"b").unwrap();
println!("load h:alpha: {:?}", sled.load("h:alpha").unwrap());
println!("exists h:beta: {}", sled.exists("h:beta").unwrap());
println!("list h:*: {:?}", sled.list("h:").unwrap());
println!("delete h:beta -> {}; exists -> {}",
    sled.delete("h:beta").unwrap(), sled.exists("h:beta").unwrap());


load h:alpha: Some([97])
exists h:beta: true
list h:*: ["h:alpha", "h:beta"]
delete h:beta -> true; exists -> false


---
## The content-addressed loop

data → CID → key → store → load — the identity of the data is its address.


In [7]:
let blob = b"the quick brown fox";
let cid: Cid = SledStorage::compute_cid(blob);
let key = cid.to_string(); // h:<sha1>
sled.store(&key, blob).unwrap();
let loaded = sled.load(&key).unwrap().unwrap();
println!("cid key:  {}", key);
println!("roundtrip byte-identical: {}", loaded == blob);
println!("exists by CID key: {}", sled.exists(&key).unwrap());


cid key:  h:ced71fa7235231bed383facfdc41c4ddcc22ecf1
roundtrip byte-identical: true
exists by CID key: true


---
## One trait, two backends

The `Storage` trait is the boundary — switching backends is a one-line change.


In [8]:
fn roundtrip<S: Storage>(s: &S) -> String {
    s.store("h:x", b"payload").unwrap();
    let v = s.load("h:x").unwrap().unwrap();
    String::from_utf8(v).unwrap()
}
println!("MemoryStorage: {}", roundtrip(&mem));
println!("SledStorage:   {}", roundtrip(&sled));


MemoryStorage: payload
SledStorage:   payload


---
## Summary

The IPFS loose end is closed: content addressing is embedded (`hllset-cid`),
the default backend is embedded (`SledStorage` + sled), and both sit behind
the same `Storage` trait. A distributed/ipfrs backend can be added later as
another trait implementation — nothing above the trait changes.
